# CPE 310 — Group 7: P2P File Transfer & Integrity Verification
**Federal University Oye-Ekiti (FUOYE)**

| | |
|---|---|
| **Course** | CPE 310 — Object-Oriented Programming |
| **Project** | P2P File Transfer & Integrity Verification System |
| **Group** | Group 7 |

---
### Notebook Structure
1. **PART A** — Project Setup (clone repo, install dependencies)
2. **PART B** — Source Code (all implementation files)
3. **PART C** — Automated Test Suite (45 tests with visible output)
4. **PART D** — Real-Life Simulation Demo (8 scenes)


---
## PART A — Project Setup

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/samuelomoyeni09-pixel/p2p_file_transfer.git
%cd p2p_file_transfer
!pip install -r requirements.txt -q
print('Setup complete.')

---
## PART B — Source Code
All implementation files are shown below with syntax highlighting.

### `src/exceptions.py` — Custom Exception Hierarchy

In [ ]:
from IPython.display import display, Code
display(Code(filename='src/exceptions.py', language='python'))

### `src/chunk_data.py` — Immutable Chunk with SHA-256 Verification

In [ ]:
display(Code(filename='src/chunk_data.py', language='python'))

### `src/file_metadata.py` — File Metadata & Chunk Layout

In [ ]:
display(Code(filename='src/file_metadata.py', language='python'))

### `src/swarm.py` — Swarm (Set of Peers Sharing a File)

In [ ]:
display(Code(filename='src/swarm.py', language='python'))

### `src/nodes.py` — Node Hierarchy (Abstract, DataPeer, Tracker, Seed)

In [ ]:
display(Code(filename='src/nodes.py', language='python'))

### `src/protocols.py` — Transfer Protocols (Push & Pull)

In [ ]:
display(Code(filename='src/protocols.py', language='python'))

### `src/transfer_session.py` — Transfer Session (Progress Tracking)

In [ ]:
display(Code(filename='src/transfer_session.py', language='python'))

---
## PART C — Automated Test Suite
### `tests/test_p2p.py` — 45 Test Cases

In [ ]:
display(Code(filename='tests/test_p2p.py', language='python'))

### Run All 45 Tests
The `-s` flag disables stdout capture so every `print()` inside each test is visible.

In [ ]:
!python -m pytest tests/test_p2p.py -s -v

---
## PART D — Real-Life Simulation Demo
This section runs a complete end-to-end simulation of the P2P network showing how
the system behaves with real inputs: file chunking, peer registration, clean downloads,
noisy downloads with retries, tamper detection, swarm management, and rate limiting.


In [ ]:
import hashlib, random
from src.chunk_data import ChunkData
from src.exceptions import ChecksumMismatchError, PeerNotFoundError
from src.file_metadata import FileMetadata
from src.nodes import DataPeerNode, MetadataTrackerNode, SeedNode
from src.protocols import PullProtocol, PushProtocol
from src.transfer_session import TransferSession

SEP  = '=' * 62
SEP2 = '-' * 62

def bar(pct, width=28):
    filled = int(width * pct / 100)
    return '[' + '#' * filled + '-' * (width - filled) + f'] {pct:.1f}%'

def section(title):
    print(f'\n{SEP}\n  {title}\n{SEP}')

print('Imports ready.')

### Scene 1 — File Creation & Metadata

In [ ]:
section('SCENE 1: File Creation & Metadata')

FILE_CONTENT = (
    b'CPE 310 Group 7 - P2P File Transfer System\n'
    b'Federal University Oye-Ekiti (FUOYE)\n'
    b'This file is split into chunks and transferred across peers.\n'
    b'Each chunk is verified with SHA-256 to detect any corruption.\n'
) * 10   # ~1.6 KB

CHUNK_SIZE = 256
metadata = FileMetadata('lecture_notes.txt', FILE_CONTENT, chunk_size_bytes=CHUNK_SIZE)

print(f'  Filename    : {metadata.filename}')
print(f'  File size   : {metadata.size_bytes} bytes')
print(f'  Chunk size  : {metadata.chunk_size_bytes} bytes')
print(f'  Num chunks  : {metadata.chunk_count}')
print(f'  SHA-256     : {metadata.sha256_hash[:40]}...')
print(f'\n  str()  -> {metadata}')
print(f'  repr() -> {repr(metadata)}')

### Scene 2 — Seed Node Shares the File

In [ ]:
section('SCENE 2: Seed Node Shares File')

seed = SeedNode('seed-01', '192.168.1.1:9000', bandwidth=100.0, max_bandwidth_bps=500_000)
seed.share_file(FILE_CONTENT, 'lecture_notes.txt', chunk_size=CHUNK_SIZE)

print(f'  Node ID     : {seed.node_id}')
print(f'  Address     : {seed.address}')
print(f'  Bandwidth   : {seed.bandwidth} Mbps')
print(f'  Upload cap  : {seed.max_bandwidth_bps} B/s')
print(f'  Files held  : {len(seed.local_files)}')
print(f'  has_file()  : {seed.has_file(metadata.sha256_hash)}')
print(f'\n  str()  -> {seed}')

### Scene 3 — Tracker Registers Peers & Ranks by Bandwidth

In [ ]:
section('SCENE 3: Tracker Registers Peers')

tracker = MetadataTrackerNode('tracker-01', '192.168.1.0:6969')
extra_peers = []

for pid, addr, bw in [('peer-A','192.168.1.10:8000',80.0),
                       ('peer-B','192.168.1.11:8001',45.0),
                       ('peer-C','192.168.1.12:8002',30.0)]:
    p = DataPeerNode(pid, addr, bandwidth=bw)
    p.share_file(FILE_CONTENT, 'lecture_notes.txt', chunk_size=CHUNK_SIZE)
    tracker.register_file(metadata, p)
    extra_peers.append(p)
    print(f'  Registered : {pid}  [{addr}]  {bw} Mbps')

tracker.register_file(metadata, seed)
print(f'  Registered : {seed.node_id}  [{seed.address}]  {seed.bandwidth} Mbps')

print(f'\n  Peer ranking by bandwidth (fastest first):')
for i, p in enumerate(tracker.find_peers(metadata.sha256_hash), 1):
    print(f'    {i}. {p.node_id}  ({p.bandwidth} Mbps)')

### Scene 4 — Clean Download via PushProtocol (0% Corruption)

In [ ]:
section('SCENE 4: Clean Download — PushProtocol (0% corruption)')

downloader = DataPeerNode('peer-NEW', '192.168.1.99:8080', bandwidth=60.0)
best_peer  = tracker.find_peers(metadata.sha256_hash)[0]
session    = TransferSession(downloader, best_peer, metadata)
proto      = PushProtocol(corruption_probability=0.0)

print(f'  Downloader : {downloader.node_id}')
print(f'  Provider   : {best_peer.node_id}')
print(f'  Protocol   : {proto}')
print()

proto.initiate(session)
for idx in range(metadata.chunk_count):
    chunk = proto.transfer_chunk(session, idx, provider=best_peer)
    session.mark_received(chunk)
    print(f'  Chunk {idx+1:>2}/{metadata.chunk_count}  '
          f'size={len(chunk.data):>3}B  '
          f'checksum={chunk.checksum[:12]}...  '
          f'verify={chunk.verify()}  '
          f'{bar(session.progress_pct)}')

print(f'\n  Transfer complete : {session.is_complete()}')
print(f'  Whole-file SHA-256 match : {proto.finalise(session)}')
print(f'  Retries needed : {session.retry_count}')

### Scene 5 — Noisy Network: PullProtocol with 40% Corruption & Automatic Retries

In [ ]:
section('SCENE 5: Noisy Network — PullProtocol (40% corruption + retries)')

random.seed(7)
downloader2 = DataPeerNode('peer-STUDENT', '192.168.1.50:9090', bandwidth=55.0)
providers   = tracker.find_peers(metadata.sha256_hash)
session2    = TransferSession(downloader2, providers[0], metadata)
proto2      = PullProtocol(corruption_probability=0.4)

print(f'  Downloader : {downloader2.node_id}')
print(f'  Providers  : {[p.node_id for p in providers]}')
print(f'  Protocol   : {proto2}')
print()

proto2.initiate(session2)
total_retries = 0

for chunk_idx in range(metadata.chunk_count):
    attempt = 0
    for provider in providers * 50:
        try:
            chunk = proto2.transfer_chunk(session2, chunk_idx, provider=provider)
            session2.mark_received(chunk)
            status = 'OK' if attempt == 0 else f'OK (after {attempt} retr{"y" if attempt==1 else "ies"})'
            print(f'  Chunk {chunk_idx+1:>2}/{metadata.chunk_count}  [{provider.node_id}]  '
                  f'{status:<24}  {bar(session2.progress_pct)}')
            break
        except ChecksumMismatchError as e:
            total_retries += 1
            attempt += 1
            session2.increment_retry()
            print(f'  Chunk {chunk_idx+1:>2}/{metadata.chunk_count}  [{provider.node_id}]  '
                  f'CORRUPTED!  expected={e.expected[:8]}...  got={e.received[:8]}...  Retrying...')

print(f'\n  Transfer complete : {session2.is_complete()}')
print(f'  Whole-file SHA-256 match : {proto2.finalise(session2)}')
print(f'  Total retries : {total_retries}')

### Scene 6 — Integrity Check: Tampered Chunk Detected

In [ ]:
section('SCENE 6: Integrity Verification — Tampered Chunk Detected')

original = ChunkData(metadata.sha256_hash, 0, FILE_CONTENT[:CHUNK_SIZE])
tampered = ChunkData._from_raw(
    metadata.sha256_hash, 0,
    b'\x00' * CHUNK_SIZE,   # corrupted data
    original.checksum        # original checksum kept — so verify() will fail
)

print(f'  Original chunk[0]  verify() = {original.verify()}   <- data intact')
print(f'  Tampered chunk[0]  verify() = {tampered.verify()}  <- data was modified!')
print()
print(f'  Stored checksum : {original.checksum[:40]}...')
print(f'  Actual data hash: {hashlib.sha256(tampered.data).hexdigest()[:40]}...')
print()
print(f'  Conclusion: checksum mismatch detected -> ChecksumMismatchError would be raised.')

### Scene 7 — Swarm Membership & Lookup

In [ ]:
section('SCENE 7: Swarm Membership')

swarm = tracker.get_swarm(metadata.sha256_hash)
print(f'  Swarm: {swarm}')
print(f'  Size  : {len(swarm)} peers')
print(f'  Members:')
for p in sorted(swarm, key=lambda x: x.bandwidth, reverse=True):
    print(f'    - {p.node_id:<12} [{p.address}]   {p.bandwidth} Mbps')

print()
print(f"  'seed-01' in swarm  -> {'seed-01' in swarm}")
print(f"  'peer-A'  in swarm  -> {'peer-A'  in swarm}")
print(f"  'ghost'   in swarm  -> {'ghost'   in swarm}")

### Scene 8 — Rate Limiter: Upload Cap Enforcement

In [ ]:
section('SCENE 8: Rate Limiter — Upload Cap Enforcement')

limited_seed = SeedNode('seed-limited', '10.0.0.1:9000', max_bandwidth_bps=400)
limited_seed.share_file(b'X' * 1000, 'bigfile.bin', chunk_size=100)
lmeta = FileMetadata('bigfile.bin', b'X' * 1000, 100)

print(f'  Upload cap  : {limited_seed.max_bandwidth_bps} B/s')
print(f'  Chunk size  : 100 B each')
print()

for i in range(6):
    chunk = limited_seed.request_chunk(lmeta.sha256_hash, i)
    used  = limited_seed.bytes_sent_this_second
    can   = limited_seed.can_send(len(chunk.data))
    print(f'  Chunk[{i}] sent   bytes_used={used:>4}B / {limited_seed.max_bandwidth_bps}B   '
          f'can_send_next_100B = {can}')
    if not can:
        print(f'  >>> Rate limit reached! Resetting (simulating next second...)')
        limited_seed.reset_bandwidth()
        print(f'  >>> bytes_used reset to 0. Continuing...\n')

### Simulation Summary

In [ ]:
print(SEP)
print('  SIMULATION COMPLETE — Group 7 P2P System')
print(SEP)
print('  Scene 1 : File metadata created and inspected             [DONE]')
print('  Scene 2 : Seed node registered file and serves chunks     [DONE]')
print(f'  Scene 3 : Tracker registered 4 peers, ranked by BW       [DONE]')
print('  Scene 4 : Clean PushProtocol download — 0% corruption     [DONE]')
print(f'  Scene 5 : PullProtocol with 40% noise — retried {total_retries}x      [DONE]')
print('  Scene 6 : Tampered chunk caught by SHA-256 verify()       [DONE]')
print('  Scene 7 : Swarm membership and lookup verified            [DONE]')
print('  Scene 8 : Rate limiter enforced upload cap                [DONE]')
print(SEP)